# ETHICS Cleaning

This notebook inspects the raw Hendrycks ETHICS files, shows cleaning diagnostics,
visualizes class/split coverage, and builds the standardized `ethics.csv` / `ethics.jsonl` outputs.


In [ ]:
from __future__ import annotations

from pathlib import Path
from ast import literal_eval
import csv
import hashlib
import json
import re
import shutil

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError("pandas is required to use this cleaning notebook.") from exc

from IPython.display import display



sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate
    return Path.cwd().resolve()


ROOT = find_project_root()
DATA_ROOT = ROOT / "Data" if (ROOT / "Data").exists() else ROOT / "data"
RAW_DIR = DATA_ROOT / "raw/hendryicks-ethics"
OUT_DIR = DATA_ROOT / "processed" / "ethics"
SAVE_OUTPUTS = False

print("Project root:", ROOT)
print("Raw dir:", RAW_DIR)
print("Output dir:", OUT_DIR)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_for_csv(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_jsonl(path: Path, rows) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows, fieldnames=None) -> None:
    ensure_dir(path.parent)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    seen.add(key)
                    fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: normalize_for_csv(row.get(k)) for k in fieldnames})


def plot_count(series, title: str, top_n: int = 15):
    counts = series.fillna("<missing>").astype(str).value_counts().head(top_n)
    if counts.empty:
        print(f"No values available for {title}")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def plot_text_length(series, title: str):
    lengths = series.fillna("").astype(str).str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(lengths, bins=30, ax=ax, color="#55A868")
    ax.set_title(title)
    ax.set_xlabel("characters")
    plt.tight_layout()
    plt.show()


In [ ]:
TEXT_FIELDS = ["input", "scenario", "prompt", "question", "text", "sentence", "story", "statement"]
LABEL_FIELDS = ["label", "answer", "gold", "gold_label", "target", "output"]


def split_from_name(name: str) -> tuple[str, str]:
    for suffix in ["train", "test_hard", "test", "ambig"]:
        needle = f"_{suffix}"
        if name.endswith(needle):
            return name[: -len(needle)], suffix
    return name, ""


raw_frames = []
for path in sorted(RAW_DIR.rglob("*.csv")):
    rel = path.relative_to(RAW_DIR)
    category = rel.parts[0]
    task, split = split_from_name(path.stem)
    df = pd.read_csv(path)
    df["source_file"] = str(rel)
    df["category"] = category
    df["task"] = task
    df["split"] = split
    raw_frames.append(df)

raw_df = pd.concat(raw_frames, ignore_index=True, sort=False)
print("Raw shape:", raw_df.shape)
display(raw_df.head())
display(pd.DataFrame({"column": raw_df.columns, "missing": raw_df.isna().sum().values}).sort_values("missing", ascending=False).head(15))


In [ ]:
def pick_first_value(row, candidates):
    for key in candidates:
        value = row.get(key)
        if pd.notna(value) and str(value).strip():
            return key, str(value).strip()
    return "", ""


analysis_rows = []
for row in raw_df.to_dict(orient="records"):
    text_key, text_value = pick_first_value(row, TEXT_FIELDS)
    label_key, label_value = pick_first_value(row, LABEL_FIELDS)
    analysis_rows.append({
        "category": row["category"],
        "task": row["task"],
        "split": row["split"],
        "source_file": row["source_file"],
        "text_field": text_key,
        "label_field": label_key,
        "text": text_value,
        "label": label_value,
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df["text_len"] = analysis_df["text"].str.len()
analysis_df["text_hash"] = analysis_df["text"].map(lambda x: hashlib.sha1(x.encode("utf-8", errors="ignore")).hexdigest() if x else "")

display(analysis_df.head())
display(analysis_df[["text", "label"]].isna().sum())
display(analysis_df[["text", "label"]].eq("").sum().rename("blank_count").to_frame())

plot_count(analysis_df["category"], "ETHICS rows by category")
plot_count(analysis_df["split"], "ETHICS rows by split")
plot_count(analysis_df.loc[analysis_df["label"] != "", "label"], "ETHICS label distribution")
plot_text_length(analysis_df["text"], "ETHICS text length distribution")

dup_counts = analysis_df.loc[analysis_df["text_hash"] != "", "text_hash"].value_counts()
print("Duplicate text hashes with more than one occurrence:", int((dup_counts > 1).sum()))


In [ ]:
cleaned_df = analysis_df.loc[:, ["text", "label", "task", "split", "source_file", "category", "text_field", "label_field", "text_hash"]].copy()
cleaned_df.insert(2, "dataset", "ethics")
cleaned_df["metadata"] = cleaned_df.apply(
    lambda row: {
        "category": row["category"],
        "text_field": row["text_field"],
        "label_field": row["label_field"],
        "text_hash": row["text_hash"],
    },
    axis=1,
)
cleaned_df = cleaned_df[["text", "label", "dataset", "task", "split", "source_file", "metadata"]]

summary_df = (
    analysis_df.assign(has_text=analysis_df["text"] != "", has_label=analysis_df["label"] != "")
    .groupby(["category", "task", "split", "source_file"], dropna=False)
    .agg(
        rows=("text", "size"),
        missing_text=("has_text", lambda s: int((~s).sum())),
        missing_label=("has_label", lambda s: int((~s).sum())),
        avg_text_len=("text_len", "mean"),
    )
    .reset_index()
)

dup_report_df = (
    analysis_df.loc[analysis_df["text_hash"] != "", ["text_hash", "category", "task", "split", "source_file", "text_field"]]
    .groupby("text_hash")
    .agg(
        occurrences=("text_hash", "size"),
        locations=("source_file", lambda s: list(s)),
    )
    .reset_index()
)
dup_report_df = dup_report_df.loc[dup_report_df["occurrences"] > 1]

print("Cleaned shape:", cleaned_df.shape)
display(cleaned_df.head())
display(summary_df.head())
display(dup_report_df.head())


In [ ]:
if SAVE_OUTPUTS:
    write_jsonl(OUT_DIR / "ethics.jsonl", cleaned_df.to_dict(orient="records"))
    write_csv(OUT_DIR / "ethics.csv", cleaned_df.to_dict(orient="records"))
    write_jsonl(OUT_DIR / "summary.jsonl", summary_df.to_dict(orient="records"))
    write_csv(OUT_DIR / "summary.csv", summary_df.to_dict(orient="records"))
    write_jsonl(OUT_DIR / "dup_report.jsonl", dup_report_df.to_dict(orient="records"))
    write_csv(OUT_DIR / "dup_report.csv", dup_report_df.to_dict(orient="records"))
    label_summary = [{"labels": cleaned_df.loc[cleaned_df["label"] != "", "label"].value_counts().to_dict()}]
    write_jsonl(OUT_DIR / "label_summary.jsonl", label_summary)
    write_csv(OUT_DIR / "label_summary.csv", label_summary)
    print("Wrote cleaned ETHICS outputs to", OUT_DIR)
else:
    print("Preview only. Set SAVE_OUTPUTS = True and rerun this cell to write cleaned files.")
